# F30K_T1S1_Dense: Calibration Lock

Khóa 128 ảnh calibration và 32 ảnh parity holdout độc lập từ Validation trước lượng tử hóa.

## Input bắt buộc

1. Output MOB_F30K_T1S1_Dense_Entry đã chạy lại bằng notebook đã sửa
2. Output MOB_F30K_T1S1_Dense_ONNX_Preflight đã chạy lại bằng notebook đã sửa
3. Output F30K_Data

Runtime: **CPU / Accelerator None / Internet Off**. Không huấn luyện, không chọn lại checkpoint và không đọc Test.


In [1]:

import csv
import hashlib
import json
import os
import shutil
import statistics
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))

def now():
    return datetime.now(timezone.utc).isoformat()

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

def unique_lock(suffix, model_id):
    matches = []
    for path in INPUT_ROOT.rglob(suffix):
        if path.is_file():
            payload = read_json(path)
            if payload.get("model_id") == model_id:
                matches.append((path, payload))
    assert len(matches) == 1, (suffix, model_id, [str(p) for p, _ in matches])
    return matches[0]

def file_by_hash(name, expected):
    matches = [p for p in INPUT_ROOT.rglob(name) if p.is_file() and sha256(p) == expected]
    assert len(matches) == 1, (name, expected, [str(p) for p in matches])
    return matches[0]

CFG = json.loads('{"short": "F30K_T1S1_Dense", "teacher": "ExpansionNet v2", "student": "MobileNetV3-Large", "student_id": "S1", "dataset": "Flickr30K", "method": "Dense-Mapped Logit-KD", "seed": 123, "vocab_name": "f30k_vocabulary_minfreq5.json", "vocab_key": "tokens", "vocab_size": 7456, "encoder_key": "encoder_state_dict", "decoder_key": "decoder_state_dict", "source_dim": 960, "encoder_params": 3231024, "decoder_params": 5084960, "decoder_batch": 1, "mask_dtype": "bool", "encoder_input": "images", "encoder_output": "memory", "token_input": "token_ids", "memory_input": "memory", "mask_input": "attention_mask", "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225], "validation_file": "f30k_validation.parquet"}')
EXPERIMENT_ID = "MOB_" + CFG["short"] + "_Calib_Lock"
OUT = Path(os.environ.get("KLCN_OUTPUT_ROOT", "/kaggle/working")) / EXPERIMENT_ID
OUT.mkdir(parents=True, exist_ok=True)

entry_path, entry = unique_lock("*_ENTRY_LOCK.json", CFG["short"])
preflight_path, preflight = unique_lock("*_ONNX_PREFLIGHT.json", CFG["short"])
assert entry["status"] == "locked" and preflight["status"] == "passed"
assert entry["selected_seed"] == CFG["seed"] == preflight["selected_seed"]
assert entry["test_set_used"] is False and preflight["test_set_used"] is False

validation_matches = sorted(p for p in INPUT_ROOT.rglob(CFG["validation_file"]) if p.is_file())
assert len(validation_matches) == 1, validation_matches
validation_path = validation_matches[0]
import pandas as pd
validation = pd.read_parquet(validation_path, columns=["image_id"])
image_ids = sorted(set(int(value) for value in validation["image_id"].tolist()))
expected_images = 5000 if CFG["dataset"] == "MS COCO 2014" else 1014
assert len(image_ids) == expected_images, (len(image_ids), expected_images)

ranked = sorted(
    image_ids,
    key=lambda image_id: hashlib.sha256(
        f"{CFG['short']}|{CFG['seed']}|{image_id}".encode("utf-8")
    ).hexdigest(),
)
calibration_ids = ranked[:128]
parity_holdout_ids = ranked[128:160]
assert len(set(calibration_ids) & set(parity_holdout_ids)) == 0

payload = {
    "schema_version": "mobile_calibration_lock_v1",
    "created_at_utc": now(),
    "status": "locked",
    "experiment_id": EXPERIMENT_ID,
    "model_id": CFG["short"],
    "teacher": CFG["teacher"],
    "student": CFG["student"],
    "dataset": CFG["dataset"],
    "method": CFG["method"],
    "selected_seed": CFG["seed"],
    "source_split": "validation",
    "validation_images": expected_images,
    "calibration_count": len(calibration_ids),
    "parity_holdout_count": len(parity_holdout_ids),
    "calibration_image_ids": calibration_ids,
    "parity_holdout_image_ids": parity_holdout_ids,
    "calibration_ids_sha256": hashlib.sha256(
        json.dumps(calibration_ids, separators=(",", ":")).encode("utf-8")
    ).hexdigest(),
    "parity_holdout_ids_sha256": hashlib.sha256(
        json.dumps(parity_holdout_ids, separators=(",", ":")).encode("utf-8")
    ).hexdigest(),
    "validation_source_sha256": sha256(validation_path),
    "disjoint": True,
    "test_set_used": False,
    "authorized_next": "INT8_Preflight_after_ONNX_FP32",
}
lock_path = OUT / (EXPERIMENT_ID + ".json")
write_json(lock_path, payload)
with (OUT / "MANIFEST.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["relative_path", "bytes", "sha256"])
    writer.writeheader()
    writer.writerow({
        "relative_path": lock_path.name,
        "bytes": lock_path.stat().st_size,
        "sha256": sha256(lock_path),
    })

print("Calibration lock        : PASSED")
print("Model                   :", CFG["teacher"], "->", CFG["student"])
print("Dataset / KD            :", CFG["dataset"], "/", CFG["method"])
print("Validation images       :", expected_images)
print("Calibration / holdout   :", len(calibration_ids), "/", len(parity_holdout_ids))
print("Test                    : SEALED / NOT READ")


Calibration lock        : PASSED
Model                   : ExpansionNet v2 -> MobileNetV3-Large
Dataset / KD            : Flickr30K / Dense-Mapped Logit-KD
Validation images       : 1014
Calibration / holdout   : 128 / 32
Test                    : SEALED / NOT READ
